**Install:** `pip install -U langchain langchain-openai langgraph`

# 🎯 1. Introduction to AI Agents

Welcome to the **AI Agents Learning Guide**. In this first notebook we lay the foundation for everything that follows.

We will cover:

1. **What is an AI Agent?** — definition and core intuition
2. **Historical context** — from rule-based systems to LLM-powered agents
3. **Agent taxonomy** — the spectrum of agent complexity
4. **Core components** — brain, tools, memory, and planning
5. **The Agent Loop** — Observe → Think → Act
6. **When to use agents** — agents vs prompting vs RAG

By the end of this notebook you will have a clear mental model of what AI Agents are, why they matter, and how they differ from simple LLM applications.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"

print(f'Model: {LLM_MODEL}')
print(f'API key configured: {bool(os.getenv("OPENAI_API_KEY"))}')

Model: gpt-4o-mini
API key configured: True


## 1.1 What Is an AI Agent?

An **AI Agent** is a system that can **perceive** its environment, **reason** about what to do, and **take actions** to achieve a goal — autonomously.

The key distinction from a regular LLM call:

| Aspect | Simple LLM Call | AI Agent |
|--------|----------------|----------|
| **Interaction** | Single request → response | Multi-step loop |
| **Tools** | None | Can call functions, APIs, databases |
| **Memory** | Stateless | Maintains context across steps |
| **Autonomy** | User drives everything | Agent decides next action |
| **Planning** | None | Breaks tasks into sub-tasks |

The simplest definition:

> **AI Agent = LLM + Tools + Memory + Reasoning Loop**

Think of it like this: a regular LLM is a **brain in a jar** — it can think, but it cannot act. An agent gives that brain **hands** (tools), **eyes** (observations), and a **notebook** (memory).

### The Simplest Agent in Pseudocode

Before we write real code, let's understand the core pattern:

In [2]:
# Import ast to safely parse Python expressions into syntax trees
# Import operator to map allowed arithmetic nodes to real functions.
import ast, operator
import ast
import operator


def safe_calculator(expression: str) -> str:
    """
    Safely evaluate simple arithmetic expressions without using eval().
    Supports:
    +, -, *, /, //, %, **, parentheses, unary + and -
    """

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos,
    }

    def evaluate(node):
        # Αν ο κόμβος είναι ολόκληρη έκφραση, αξιολογούμε το κύριο σώμα της έκφρασης.
        if isinstance(node, ast.Expression):
            return evaluate(node.body)

        # Επιτρέπουμε μόνο αριθμητικές σταθερές, δηλαδή int και float.
        # Απορρίπτουμε strings, booleans, None κ.λπ.
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise ValueError("Only numbers are allowed.")

        # Αν ο κόμβος είναι δυαδική πράξη, π.χ. 2 + 3, 10 * 5, 8 / 2.
        if isinstance(node, ast.BinOp):
            # Υπολογίζουμε αναδρομικά το αριστερό και το δεξί μέρος της πράξης.
            left = evaluate(node.left)
            right = evaluate(node.right)

            # Παίρνουμε τον τύπο του τελεστή, π.χ. ast.Add, ast.Sub, ast.Mult.
            operator_type = type(node.op)

            # Ελέγχουμε αν ο τελεστής ανήκει στους επιτρεπτούς τελεστές.
            if operator_type not in allowed_operators:
                raise ValueError("Operator not allowed.")

            # Εκτελούμε την πραγματική αριθμητική πράξη μέσω του dictionary allowed_operators.
            return allowed_operators[operator_type](left, right)

        # Αν ο κόμβος είναι μοναδιαία πράξη, π.χ. -5 ή +10.
        if isinstance(node, ast.UnaryOp):
            # Υπολογίζουμε αναδρομικά τον αριθμό στον οποίο εφαρμόζεται το πρόσημο.
            operand = evaluate(node.operand)

            # Παίρνουμε τον τύπο του μοναδιαίου τελεστή, π.χ. ast.USub ή ast.UAdd.
            operator_type = type(node.op)

            # Ελέγχουμε αν ο μοναδιαίος τελεστής επιτρέπεται.
            if operator_type not in allowed_operators:
                raise ValueError("Unary operator not allowed.")

            # Εφαρμόζουμε τον τελεστή στο operand.
            return allowed_operators[operator_type](operand)

        # Αν φτάσουμε εδώ, σημαίνει ότι βρέθηκε μη επιτρεπτός κόμβος,
        # όπως function call, variable name, attribute access κ.λπ.
        raise ValueError("Invalid expression.")


    try:
        # Μετατρέπουμε το string της αριθμητικής έκφρασης σε AST.
        # Το mode="eval" επιτρέπει μόνο μία έκφραση και όχι κανονικό Python script.
        parsed_expression = ast.parse(expression, mode="eval")

        # Αξιολογούμε με ασφάλεια το AST χρησιμοποιώντας τη δική μας evaluate().
        result = evaluate(parsed_expression)

        # Επιστρέφουμε το αποτέλεσμα ως string, ώστε να ταιριάζει με το interface του tool.
        return str(result)

    except ZeroDivisionError:
        # Ειδική διαχείριση για διαίρεση με το μηδέν.
        return "Error: Division by zero."

    except Exception as error:
        # Γενική διαχείριση σφαλμάτων για μη έγκυρες ή μη επιτρεπτές εκφράσεις.
        return f"Error: {error}"


def simplest_agent(user_query: str) -> str:
    """
    The most basic agent loop:
    1. Receive input
    2. Think about what to do
    3. Optionally use a tool
    4. Return a response
    """

    tools = {
        "calculator": safe_calculator,
        "greet": lambda name: f"Hello, {name}! How can I help you today?"
    }

    # 1. THINK — decide if we need a tool
    if any(op in user_query for op in ["+", "-", "*", "/", "%"]):
        action = "calculator"
        action_input = user_query

    elif "hello" in user_query.lower():
        action = "greet"
        action_input = "User"

    else:
        return f"I received your query: {user_query}"

    # 2. ACT — execute the tool
    result = tools[action](action_input)

    # 3. OBSERVE — return the result
    return f"Tool [{action}] returned: {result}"


# Test it
print(simplest_agent("2 + 3 * 4"))
print(simplest_agent("hello there"))
print(simplest_agent("What is AI?"))
print(simplest_agent("(10 + 5) / 3"))
print(simplest_agent("2 ** 8"))
print(simplest_agent("__import__('os').system('rm -rf /')"))

Tool [calculator] returned: 14
Tool [greet] returned: Hello, User! How can I help you today?
I received your query: What is AI?
Tool [calculator] returned: 5.0
Tool [calculator] returned: 256
Tool [calculator] returned: Error: Invalid expression.


## 1.2 Historical Context — From Rules to LLMs

AI agents are not a new idea. What changed is the **reasoning engine** at their core.

### Evolution of Agent Architectures

| Era | Agent Type | Reasoning Engine | Limitation |
|-----|-----------|-----------------|------------|
| **1990s** | Rule-based (expert systems) | If-then rules | Brittle, hard to maintain |
| **2000s** | Reinforcement Learning agents | Value functions, policies | Needs massive training data |
| **2010s** | Deep RL (AlphaGo, Atari) | Neural networks | Domain-specific, expensive |
| **2023+** | LLM-powered agents | Large Language Models | Context limits, hallucination |

The breakthrough of **LLM-powered agents** is that the language model can:
- Understand natural language instructions
- Reason about complex tasks in plain text
- Generate structured tool calls (function calling)
- Adapt to new tasks without retraining

This makes LLMs the first **general-purpose reasoning engine** suitable for building agents across domains.

## 1.3 Agent Taxonomy — The Spectrum of Complexity

Not all agents are equal. There is a **spectrum of agency** from simple to complex:

| Level | Type | Description | Example |
|-------|------|-------------|--------|
| ☆☆☆ | **Simple Reflex** | Fixed rules, no memory | Thermostat |
| ★☆☆ | **Model-Based** | Internal state, environment model | GPS navigation |
| ★★☆ | **Goal-Based** | Plans actions toward a goal | Game-playing AI |
| ★★★ | **Utility-Based** | Optimizes for best outcome | Trading agent |
| ★★★+ | **Learning** | Improves from experience | Self-driving car |

In this course, we focus on **LLM-powered agents** which typically operate at the **Goal-Based** to **Learning** level, depending on their memory and planning capabilities.

### The Agency Spectrum for LLM Applications


<img src="images/agent-taxonomy.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

## 1.4 Core Components of an AI Agent

Every AI Agent, regardless of framework, consists of **four core components**:

### 1. 🧠 Brain (LLM / Reasoning Engine)
The central decision-maker. In modern agents, this is a Large Language Model. It:
- Interprets the user's request
- Decides which tool to use (or whether to answer directly)
- Generates the response

### 2. 🔧 Tools (Actions / Actuators)
External capabilities the agent can invoke:
- API calls (search, weather, databases)
- Code execution
- File operations
- Web browsing

### 3. 🧾 Memory (State)
What the agent remembers:
- **Short-term**: current conversation context
- **Long-term**: facts persisted across sessions
- **Working**: scratchpad for current task

### 4. 📋 Planning (Strategy)
How the agent breaks down complex tasks:
- Task decomposition
- Step-by-step reasoning (Chain-of-Thought)
- Self-correction and reflection

<img src="images/agent-architecture.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

## 1.5 The Agent Loop — Observe → Think → Act

The defining feature of an agent is its **loop**. Unlike a single LLM call, an agent runs in a cycle:

<img src="images/agent-loop.png" height="700px" wifth="700px" style="border-radius:10px;margin:12px 0;"/>


This is often called the **TAO loop** (Think-Act-Observe) or **ReAct loop** (Reason + Act). We will implement it fully in notebook 08.

In [3]:
import ast
import operator


allowed_operators = {
    ast.Add: operator.add,        # +
    ast.Sub: operator.sub,        # -
    ast.Mult: operator.mul,       # *
    ast.Div: operator.truediv,    # /
    ast.UAdd: operator.pos,       # +x
    ast.USub: operator.neg,       # -x
}


def evaluate(node):
    # Αν ο κόμβος είναι ολόκληρη έκφραση, αξιολογούμε το body της.
    if isinstance(node, ast.Expression):
        return evaluate(node.body)

    # Επιτρέπουμε μόνο αριθμούς: int και float.
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool):
            raise ValueError("Booleans are not allowed.")

        if isinstance(node.value, (int, float)):
            return node.value

        raise ValueError("Only numbers are allowed.")

    # Δυαδικές πράξεις: 2 + 3, 10 * 5, 8 / 2.
    if isinstance(node, ast.BinOp):
        left = evaluate(node.left)
        right = evaluate(node.right)

        operator_type = type(node.op)

        if operator_type not in allowed_operators:
            raise ValueError("Operator not allowed.")

        return allowed_operators[operator_type](left, right)

    # Μοναδιαίες πράξεις: -5, +10.
    if isinstance(node, ast.UnaryOp):
        operand = evaluate(node.operand)

        operator_type = type(node.op)

        if operator_type not in allowed_operators:
            raise ValueError("Unary operator not allowed.")

        return allowed_operators[operator_type](operand)

    # Οτιδήποτε άλλο απορρίπτεται:
    # function calls, variables, imports, attributes, lists, dicts κ.λπ.
    raise ValueError("Invalid expression.")

### Implementing a Minimal Agent Loop

https://developers.openai.com/api/docs/guides/function-calling

In [4]:
# Factory-based agent: τα 50+ lines του χειροκίνητου loop
# γίνονται μία γραμμή με τη create_agent.
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        allowed = set('0123456789+-*/.() ')
        if all(c in allowed for c in expression):
            return str(evaluate(ast.parse(expression, mode='eval')))
        return 'Error: Invalid expression'
    except Exception as e:
        return f'Error: {e}'

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# ── ΕΝΑ LINE: το αντίστοιχο όλου του agent_loop() από πάνω ──
agent = create_agent(
    model=llm,
    tools=[calculate],
    system_prompt='You are a helpful assistant. Use the calculate tool for math.',
)

result = agent.invoke({'messages': [HumanMessage(content='What is (15 * 23) + (47 * 8)?')]})
for m in result['messages']:
    role = m.__class__.__name__
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'  → tool call: {tc["name"]}({tc["args"]})')
    elif getattr(m, 'content', None):
        print(f'{role}: {m.content}')


c:\Users\user\Desktop\ai-agents-course-me\venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


HumanMessage: What is (15 * 23) + (47 * 8)?
  → tool call: calculate({'expression': '(15 * 23) + (47 * 8)'})
ToolMessage: 721
AIMessage: The result of \( (15 \times 23) + (47 \times 8) \) is 721.


### Tools with Pydantic

In [5]:
# Όταν χρησιμοποιούμε το LangChain @tool decorator, το schema
# δημιουργείται αυτόματα από τα type hints — δεν χρειαζόμαστε καν
# pydantic_function_tool. Η create_agent χειρίζεται όλο το
# tool-calling lifecycle εσωτερικά.
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.agents import create_agent

class CalculateArgs(BaseModel):
    expression: str = Field(..., description='A mathematical expression to evaluate.')

@tool('calculate', args_schema=CalculateArgs)
def calculate_pyd(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        allowed = set('0123456789+-*/.() ')
        if all(c in allowed for c in expression):
            return str(evaluate(ast.parse(expression, mode='eval')))
        return 'Error: Invalid expression'
    except Exception as e:
        return f'Error: {e}'

agent_pyd = create_agent(
    model=ChatOpenAI(model=LLM_MODEL, temperature=0),
    tools=[calculate_pyd],
    system_prompt='You are a helpful assistant. Use the calculate tool for math.',
)

result = agent_pyd.invoke({'messages': [HumanMessage(content='What is (15 * 23) + (47 * 8)?')]})
print(result['messages'][-1].content)


The result of \( (15 \times 23) + (47 \times 8) \) is 721.


## 1.6 When to Use Agents vs. Simpler Approaches

Not every problem needs an agent. Use the simplest approach that solves your problem:

| Approach | Best For | Complexity |
|----------|---------|------------|
| **Direct Prompt** | Simple Q&A, text generation | ★☆☆☆☆ |
| **Prompt + Few-shot** | Classification, formatting | ★★☆☆☆ |
| **RAG** | Knowledge-grounded answers | ★★★☆☆ |
| **Single Agent** | Multi-step tasks, tool use | ★★★★☆ |
| **Multi-Agent** | Complex workflows, specialization | ★★★★★ |

### Decision Rules

**Use an agent when:**
- The task requires **multiple steps** that depend on intermediate results
- The system needs to **interact with external tools** (APIs, databases, files)
- The task requires **dynamic decision-making** (different paths based on context)
- You need **autonomous operation** with minimal human intervention

**Don't use an agent when:**
- A single LLM call suffices (summarization, translation)
- The workflow is fully deterministic (no LLM reasoning needed)
- Latency is critical and you can't afford multiple LLM calls
- The task is too sensitive for autonomous operation

## 💡 Exercise 1: Design Your First Agent

**Task**: Design an agent architecture for a **Travel Planning Assistant**.

Answer the following questions:

1. **Brain**: What model would you use? What system prompt?
2. **Tools**: List 3-4 tools the agent would need (e.g., flight search, hotel booking)
3. **Memory**: What information should the agent remember across interactions?
4. **Loop**: Describe the step-by-step flow for the query: *"Plan a 3-day trip to Tokyo in March"*
5. **Stopping condition**: When should the agent stop and present results?

Write your design in the cell below:

In [6]:
# Exercise 1: Travel Planning Agent Design
# Fill in your design below

travel_agent_design = {
    'brain': {
        'model': 'gpt-4o-mini',  # Your choice — why?
        'system_prompt': 'You are a travel planning assistant...',  # Write a full prompt
    },
    'tools': [
        # List your tools here, e.g.:
        # {'name': 'search_flights', 'description': '...', 'parameters': [...]},
    ],
    'memory': {
        'short_term': [],  # What goes here?
        'long_term': [],   # What goes here?
    },
    'stopping_condition': '...',  # When does the agent stop?
}

# Print your design
import json
print(json.dumps(travel_agent_design, indent=2))

{
  "brain": {
    "model": "gpt-4o-mini",
    "system_prompt": "You are a travel planning assistant..."
  },
  "tools": [],
  "memory": {
    "short_term": [],
    "long_term": []
  },
  "stopping_condition": "..."
}


## 📝 Summary

In this notebook we established the foundation:

| Concept | Key Takeaway |
|---------|-------------|
| **AI Agent** | LLM + Tools + Memory + Reasoning Loop |
| **History** | LLMs are the first general-purpose reasoning engine for agents |
| **Taxonomy** | Simple Reflex → Model-Based → Goal-Based → Utility → Learning |
| **Components** | Brain (LLM), Tools (APIs), Memory (state), Planning (strategy) |
| **Agent Loop** | Observe → Think → Act → (repeat until done) |
| **When to use** | Multi-step tasks, tool use, dynamic decisions |

### What's Next

In **Notebook 02: LLMs as Reasoning Engines**, we will dive deep into how LLMs work as the *brain* of an agent — structured output, multi-provider setup, and the limitations you must understand.